# Cyclistic bike share

## Importing libraries

In [1]:
import glob
import os
import pandas as pd

## Configuration

### Set float format to 2 decimal places

In [2]:
pd.set_option('display.float_format', '{:.2f}'.format)

### Path configuration

In [3]:
input_path = "data/raw"
output_path = "data/cleaned"

## Importing data

In [4]:
files = glob.glob(f'{input_path}/*.csv')

dfs = []

for file in files:
    df = pd.read_csv(file)
    dfs.append(df)

bike_data = pd.concat(dfs, ignore_index=True)

## Basic info

In [5]:
bike_data.shape

(6037968, 13)

In [6]:
bike_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 6037968 entries, 0 to 6037967
Data columns (total 13 columns):
 #   Column              Dtype  
---  ------              -----  
 0   ride_id             str    
 1   rideable_type       str    
 2   started_at          str    
 3   ended_at            str    
 4   start_station_name  str    
 5   start_station_id    str    
 6   end_station_name    str    
 7   end_station_id      str    
 8   start_lat           float64
 9   start_lng           float64
 10  end_lat             float64
 11  end_lng             float64
 12  member_casual       str    
dtypes: float64(4), str(9)
memory usage: 598.9 MB


Renaming member_casual column to rider_type

In [7]:
bike_data.rename(columns={'member_casual': 'rider_type'}, inplace=True)

bike_data.columns

Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'rider_type'],
      dtype='str')

In [8]:
bike_data.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,rider_type
0,5C00A6B5A60E4AFC,electric_bike,2025-08-28 15:56:46.803,2025-08-28 16:11:54.890,NaN,NaN,NaN,NaN,41.88,-87.64,41.89,-87.62,casual
1,FB25D4B923656D01,electric_bike,2025-08-27 19:15:53.059,2025-08-27 19:34:13.683,NaN,NaN,NaN,NaN,41.95,-87.64,41.89,-87.61,casual
2,C665B0EC937C860C,electric_bike,2025-08-27 20:27:31.394,2025-08-27 20:38:29.890,NaN,NaN,NaN,NaN,41.92,-87.65,41.90,-87.64,casual
3,C68C54D5FD9717C2,electric_bike,2025-08-28 15:33:21.055,2025-08-28 15:39:58.052,NaN,NaN,NaN,NaN,41.89,-87.65,41.88,-87.65,casual
4,902D69C29B0E65F8,electric_bike,2025-08-28 09:59:44.846,2025-08-28 10:11:17.343,NaN,NaN,NaN,NaN,41.98,-87.68,41.95,-87.70,casual


In [9]:
bike_data["rider_type"].value_counts()

rider_type
member    3886915
casual    2151053
Name: count, dtype: int64

In [10]:
bike_data.isnull().sum()

ride_id                     0
rideable_type               0
started_at                  0
ended_at                    0
start_station_name    1273200
start_station_id      1273200
end_station_name      1336777
end_station_id        1336777
start_lat                   0
start_lng                   0
end_lat                  5436
end_lng                  5436
rider_type                  0
dtype: int64

null values are not needed to be filled

## Cleaning data

### Converting str to datetime

In [11]:
# Convert the 'started_at' column to datetime format
bike_data['started_at'] = pd.to_datetime(bike_data['started_at'])

# Convert the 'ended_at' column to datetime format
bike_data['ended_at'] = pd.to_datetime(bike_data['ended_at'])

# Check the data types of the 'started_at' and 'ended_at' columns
bike_data[['started_at', 'ended_at']].dtypes

started_at    datetime64[us]
ended_at      datetime64[us]
dtype: object

### Calculating ride duration

In [12]:
# Calculate ride duration
bike_data['ride_length_minutes'] = bike_data['ended_at'] - bike_data['started_at']

# Convert to minutes
bike_data['ride_length_minutes'] = round(bike_data['ride_length_minutes'].dt.total_seconds() / 60, 2)

# Filter out negative ride durations
bike_data = bike_data[bike_data['ride_length_minutes'] > 0]

# Check
bike_data[['ride_length_minutes']].head()

,ride_length_minutes
0,15.13
1,18.34
2,10.97
3,6.62
4,11.54


### Extracting Day, Month and Hour of the day from Datetime

In [13]:
# Add a new column for the day of the week
bike_data['day_of_week'] = bike_data['started_at'].dt.day_name()

# Add a new column for the month
bike_data['month'] = bike_data['started_at'].dt.month_name()

# Add a new column for the hour of the day
bike_data['hour'] = bike_data['started_at'].dt.hour

# Check
bike_data[['started_at', 'day_of_week', 'month', 'hour']].head()

,started_at,day_of_week,month,hour
0,2025-08-28 15:56:46.803,Thursday,August,15
1,2025-08-27 19:15:53.059,Wednesday,August,19
2,2025-08-27 20:27:31.394,Wednesday,August,20
3,2025-08-28 15:33:21.055,Thursday,August,15
4,2025-08-28 09:59:44.846,Thursday,August,9


### Check for unusual values and fixing them

#### ride lengths

In [14]:
bike_data['ride_length_minutes'].describe()

count   6037812.00
mean         15.55
std          52.10
min           0.01
25%           5.35
50%           9.34
75%          16.36
max        1559.95
Name: ride_length_minutes, dtype: float64

#### Check for casual riders max ride lengths

In [15]:
bike_data.groupby('rider_type')['ride_length_minutes'].max()

rider_type
casual   1559.95
member   1559.90
Name: ride_length_minutes, dtype: float64

<p style='color: green'>1559.95 minutes of ride lengths are okay for casual riders as they can have full-day passes.</p>

#### Fix duplicates

In [16]:
bike_data['ride_id'].duplicated().sum()

np.int64(35)

In [17]:
duplicates = bike_data[bike_data['ride_id'].duplicated(keep=False)]

duplicates.sort_values("ride_id")

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,rider_type,ride_length_minutes,day_of_week,month,hour
3548524,03C5375398BE0649,electric_bike,2026-04-30 23:46:36.039,2026-05-01 00:04:48.512,NaN,NaN,Clark St & Arlington Pl,CHI02109,41.90,-87.63,41.93,-87.64,member,18.21,Thursday,April,23
4289378,03C5375398BE0649,electric_bike,2026-04-30 23:46:36.039,2026-05-01 00:04:48.512,NaN,NaN,Clark St & Arlington Pl,CHI02109,41.90,-87.63,41.93,-87.64,member,18.21,Thursday,April,23
3669856,0612B2A21BA5E988,classic_bike,2026-04-30 15:56:11.305,2026-05-01 16:56:08.148,Fulton Market,CHI01880,NaN,NaN,41.89,-87.65,NaN,NaN,member,1499.95,Thursday,April,15
4012437,0612B2A21BA5E988,classic_bike,2026-04-30 15:56:11.305,2026-05-01 16:56:08.148,Fulton Market,CHI01880,NaN,NaN,41.89,-87.65,NaN,NaN,member,1499.95,Thursday,April,15
3904445,10117825A92C182D,classic_bike,2026-04-30 23:56:24.321,2026-05-01 00:01:51.860,Peoria St & Kinzie St,CHI02098,Ogden Ave & Chicago Ave,CHI00275,41.89,-87.65,41.90,-87.65,casual,5.46,Thursday,April,23
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3689425,E87E123751FA42E1,classic_bike,2026-04-30 17:27:25.585,2026-05-01 18:27:01.264,Canal St & Madison St,CHI00498,NaN,NaN,41.88,-87.64,NaN,NaN,casual,1499.59,Thursday,April,17
4206315,E923A7ACDD4A9293,classic_bike,2026-04-30 23:58:06.960,2026-05-01 00:00:34.820,Clinton St & Lake St,CHI01746,Wacker Dr & Washington St,CHI01787,41.89,-87.64,41.88,-87.64,casual,2.46,Thursday,April,23
3358204,E923A7ACDD4A9293,classic_bike,2026-04-30 23:58:06.960,2026-05-01 00:00:34.820,Clinton St & Lake St,CHI01746,Wacker Dr & Washington St,CHI01787,41.89,-87.64,41.88,-87.64,casual,2.46,Thursday,April,23
3455842,EAB7C279D822C763,classic_bike,2026-04-30 23:58:36.620,2026-05-01 00:02:45.401,Green St & Randolph St,CHI01988,Clinton St & Madison St,CHI00233,41.88,-87.65,41.88,-87.64,member,4.15,Thursday,April,23


In [18]:
bike_data = bike_data.drop_duplicates(subset="ride_id", keep="first")

bike_data['ride_id'].duplicated().sum()

np.int64(0)

## Export cleaned data

In [19]:
# Create the directory if it doesn't exist
os.makedirs(output_path, exist_ok=True)

# Save the cleaned data
bike_data.to_csv(f"{output_path}/cyclistic-bike-share-cleaned.csv", index=False)

print(f"Data cleaning completed. Cleaned data saved to '{output_path}/cyclistic-bike-share-cleaned.csv'.")

Data cleaning completed. Cleaned data saved to 'data/cleaned/cyclistic-bike-share-cleaned.csv'.
